# Course 4 — Decision Trees

Greedy recursive splitting on Gini or entropy. The single most
interpretable model in this course — every prediction is a flowchart.

**Sections**
1. Classification trees on Carseats (0:00–0:30)
2. Regression trees and pruning (0:30–1:00)
3. Strengths, weaknesses, and the road to ensembles (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, mean_squared_error
rng = np.random.default_rng(0)
carseats = load_dataset('carseats')
boston = load_dataset('boston')


## 1. Classification tree on Carseats

In [ ]:
c = carseats.copy()
c['High'] = (c['Sales'] > 8).astype(int)
c = pd.get_dummies(c.drop(columns=['Sales']), columns=['ShelveLoc','Urban','US'],
                    drop_first=True, dtype=float)
X = c.drop(columns=['High']); y = c['High']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(f'depth-3 tree test accuracy = {tree.score(Xte, yte):.4f}')


### Gini impurity

At each node, the split that maximally drops $1 - \sum_k p_k^2$ wins.
For binary classification with `p = 0.5`, Gini = 0.5 (worst). For a
pure node, Gini = 0.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=['Low','High'],
          filled=True, ax=ax, fontsize=8)
plt.show()


## 2. Regression trees and pruning

In [ ]:
Xb = boston.drop(columns=['medv']); yb = boston['medv']
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=0)
tree = DecisionTreeRegressor(random_state=0).fit(Xtr, ytr)
print(f'unpruned depth = {tree.get_depth()}, test MSE = {mean_squared_error(yte, tree.predict(Xte)):.2f}')


### Cost-complexity pruning path

In [ ]:
path = tree.cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[:-1]  # drop the trivial root-only tree
cv_mse = []
for a in alphas:
    m = DecisionTreeRegressor(random_state=0, ccp_alpha=a)
    cv_mse.append(-cross_val_score(m, Xtr, ytr, cv=5,
                                    scoring='neg_mean_squared_error').mean())
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, cv_mse, marker='.')
best_a = alphas[int(np.argmin(cv_mse))]
ax.axvline(best_a, color='C3', linestyle='--', label=f'best α = {best_a:.3f}')
ax.set_xlabel(r'$ccp\_alpha$'); ax.set_ylabel('5-fold CV MSE'); ax.legend()
ax.set_title('Cost-complexity pruning path'); plt.show()
pruned = DecisionTreeRegressor(random_state=0, ccp_alpha=best_a).fit(Xtr, ytr)
print(f'pruned depth={pruned.get_depth()}  test MSE = {mean_squared_error(yte, pruned.predict(Xte)):.2f}')


## 3. Strengths, weaknesses, ensembles

**Strengths.** Interpretable, no scaling required, mixed dtypes, captures
  interactions naturally.

**Weakness.** Very high variance. Refit on a bootstrap sample → totally
  different tree.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, seed in zip(axes.ravel(), [0, 1, 2, 3]):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Xtr), len(Xtr), replace=True)
    t = DecisionTreeClassifier(max_depth=3, random_state=seed)
    # use the classification problem for clarity
    Xc = c.drop(columns=['High']); yc = c['High']
    Xc_tr = Xc.iloc[idx]; yc_tr = yc.iloc[idx]
    t.fit(Xc_tr, yc_tr)
    plot_tree(t, feature_names=list(Xc.columns), class_names=['Low','High'],
              filled=True, ax=ax, fontsize=6)
    ax.set_title(f'bootstrap seed = {seed}')
plt.tight_layout(); plt.show()


Same data, four different first splits, totally different trees. This
variance is the problem ensembles solve in Course 5.

### Recap
- A tree is a sequence of axis-aligned binary splits chosen greedily
  to minimize Gini (classification) or MSE (regression).
- Prune via `ccp_alpha` to fight overfitting.
- Variance is the weakness — refit on a bootstrap, get a different tree.
  Ensembles average that variance away.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load `tips`. Create the binary target `high_tip = (tip / total_bill) > 0.18`.
Fit a `DecisionTreeClassifier(max_depth=3)` to predict it from
`total_bill, size`, one-hot `day, time, sex, smoker`. Report test accuracy.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
tips = load_dataset('tips')
tips['high_tip'] = ((tips['tip'] / tips['total_bill']) > 0.18).astype(int)
X = pd.get_dummies(tips[['total_bill','size','day','time','sex','smoker']],
                    columns=['day','time','sex','smoker'], drop_first=True, dtype=float)
y = tips['high_tip']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(f'depth-3 test accuracy = {tree.score(Xte, yte):.4f}')


---

## Exercise 2

**Task 2.** Plot the depth-3 tree. Which feature does the root split on?

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
fig, ax = plt.subplots(figsize=(11, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=['Low','High'],
          filled=True, ax=ax, fontsize=8)
plt.show()
print('root split feature:', X.columns[tree.tree_.feature[0]])


---

## Exercise 3

**Task 3.** Prune via `cost_complexity_pruning_path` + 5-fold CV.
Compare test accuracy before and after pruning.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
path = full.cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[:-1]
scores = [cross_val_score(DecisionTreeClassifier(random_state=0, ccp_alpha=a),
                           Xtr, ytr, cv=5).mean() for a in alphas]
best_a = alphas[int(np.argmax(scores))]
pruned = DecisionTreeClassifier(random_state=0, ccp_alpha=best_a).fit(Xtr, ytr)
print(f'unpruned depth={full.get_depth()}  test acc={full.score(Xte, yte):.4f}')
print(f'pruned   depth={pruned.get_depth()}  test acc={pruned.score(Xte, yte):.4f}')
